# Лабораторная работа №2: «Разработка сверточной нейронной сети с использованием PyTorch»

**Выполнил:** Колесников Д.Е., группа ______

**Цель:** разработать и обучить сверточную нейронную сеть для классификации изображений CIFAR-10.

**Решение кратко:**
- архитектура — собственная реализация сети в стиле **ResNet-9** (8 сверточных слоёв + 1 полносвязный, два остаточных блока) из базовых слоёв `torch.nn`;
- аугментация — случайный сдвиг (crop с отражённой рамкой 4 px), горизонтальное отражение и Cutout (случайное стирание фрагмента);
- функция потерь — кросс-энтропия со сглаживанием меток (label smoothing = 0.1);
- оптимизатор — SGD с моментом Нестерова и weight decay, расписание скорости обучения One-Cycle;
- обучение в смешанной точности (AMP) на GPU;
- на тесте дополнительно — усреднение предсказаний по изображению и его зеркальной копии (TTA).

> Ноутбук рассчитан на запуск на GPU (Google Colab / Kaggle: *Runtime → Change runtime type → GPU*).
> Обучение 50 эпох на T4 занимает ~15–20 минут. На CPU код тоже работает, но обучение займёт много часов.

## 1. Импорт библиотек и настройка окружения

In [ ]:
import os
import time
import random

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as T
from torch.utils.data import DataLoader

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"
torch.backends.cudnn.benchmark = True   # cuDNN подбирает самый быстрый алгоритм свёртки под размеры входа

print("PyTorch:", torch.__version__, "| torchvision:", torchvision.__version__)
print("Устройство:", DEVICE, f"({torch.cuda.get_device_name(0)})" if DEVICE.type == "cuda" else "")

### Гиперпараметры

In [ ]:
DATA_DIR = "./data"
BATCH_SIZE = 256
EPOCHS = 50
MAX_LR = 0.1            # пик скорости обучения в One-Cycle
MOMENTUM = 0.9
WEIGHT_DECAY = 5e-4
LABEL_SMOOTHING = 0.1
NUM_WORKERS = min(4, os.cpu_count() or 1)

## 2. Загрузка данных

CIFAR-10: 60 000 цветных изображений $32\times32$, 10 классов, 50 000 — обучающая выборка, 10 000 — тестовая.
Загрузку выполняет `torchvision.datasets.CIFAR10` (это только загрузчик данных, а не готовая модель).

In [ ]:
train_raw = torchvision.datasets.CIFAR10(DATA_DIR, train=True, download=True)
test_raw = torchvision.datasets.CIFAR10(DATA_DIR, train=False, download=True)
CLASSES = train_raw.classes

print(f"Обучающая выборка: {len(train_raw)} изображений, массив {train_raw.data.shape}, dtype={train_raw.data.dtype}")
print(f"Тестовая выборка:  {len(test_raw)} изображений, массив {test_raw.data.shape}")
print(f"Форма одного изображения: {train_raw.data.shape[1:]} (H, W, C) -> тензор (C, H, W) = (3, 32, 32)")
print(f"Метки: целые числа 0..9, массив формы ({len(train_raw.targets)},)")
print("Классы:", CLASSES)
print("Примеров на класс (train):", np.bincount(train_raw.targets, minlength=10))

In [ ]:
fig, axes = plt.subplots(3, 8, figsize=(13, 5.4))
idx = np.random.choice(len(train_raw), size=axes.size, replace=False)
for ax, i in zip(axes.flat, idx):
    ax.imshow(train_raw.data[i])
    ax.set_title(CLASSES[train_raw.targets[i]], fontsize=10)
    ax.axis("off")
plt.suptitle("Случайные изображения из обучающей выборки")
plt.tight_layout()
plt.show()

## 3. Предобработка и аугментация

1. **Нормализация.** Для каждого канала вычитаем среднее и делим на стандартное отклонение, посчитанные **по обучающей выборке**:
   входы получаются центрированными и одного масштаба, что ускоряет сходимость.
2. **Аугментация (только для обучения):**
   - `RandomCrop(32, padding=4, padding_mode="reflect")` — изображение дополняется отражённой рамкой и из него вырезается случайный фрагмент $32\times32$,
     т.е. случайный сдвиг до 4 px. Учит сеть инвариантности к положению объекта;
   - `RandomHorizontalFlip()` — зеркальное отражение (для классов CIFAR-10 смысл картинки не меняется, в отличие, например, от цифр);
   - `RandomErasing` (**Cutout**) — с вероятностью 0.5 случайный прямоугольник площадью 5–20% изображения заменяется нулём
     (после нормализации это «средний» цвет). Сеть не может опираться на один характерный фрагмент (например, только на морду кошки)
     и учится использовать весь объект; это сильный регуляризатор, полезный при длинном обучении.

   Аугментация — основной способ борьбы с переобучением: при 50 000 картинок и ~6.5 млн параметров сеть без неё быстро запоминает обучающую выборку.

In [ ]:
MEAN = (train_raw.data / 255.0).mean(axis=(0, 1, 2))
STD = (train_raw.data / 255.0).std(axis=(0, 1, 2))
print("Среднее по каналам (R, G, B):", MEAN.round(4))
print("Ст. отклонение по каналам:   ", STD.round(4))

train_tf = T.Compose([
    T.RandomCrop(32, padding=4, padding_mode="reflect"),
    T.RandomHorizontalFlip(),
    T.ToTensor(),
    T.Normalize(MEAN, STD),
    T.RandomErasing(p=0.5, scale=(0.05, 0.2), ratio=(0.5, 2.0), value=0),
])
test_tf = T.Compose([T.ToTensor(), T.Normalize(MEAN, STD)])

train_ds = torchvision.datasets.CIFAR10(DATA_DIR, train=True, transform=train_tf)
train_eval_ds = torchvision.datasets.CIFAR10(DATA_DIR, train=True, transform=test_tf)   # train без аугментации — для оценки
test_ds = torchvision.datasets.CIFAR10(DATA_DIR, train=False, transform=test_tf)

loader_kw = dict(num_workers=NUM_WORKERS, pin_memory=DEVICE.type == "cuda", persistent_workers=NUM_WORKERS > 0)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=True, **loader_kw)
train_eval_loader = DataLoader(train_eval_ds, batch_size=1024, shuffle=False, **loader_kw)
test_loader = DataLoader(test_ds, batch_size=1024, shuffle=False, **loader_kw)

xb, yb = next(iter(train_loader))
print("Пачка изображений:", tuple(xb.shape), xb.dtype, "| пачка меток:", tuple(yb.shape), yb.dtype)

In [ ]:
# как выглядит аугментация: одно и то же изображение, 8 случайных вариантов (после обратной нормализации)
img = T.ToPILImage()(train_raw.data[idx[0]])
def denorm(x):
    return (x.permute(1, 2, 0).numpy() * STD + MEAN).clip(0, 1)
fig, axes = plt.subplots(1, 9, figsize=(13, 1.8))
axes[0].imshow(img); axes[0].set_title("оригинал")
for ax in axes[1:]:
    ax.imshow(denorm(train_tf(img)))
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()

## 4. Архитектура сети

### 4.1. Сверточный слой (нотация лекции)

Типовой сверточный слой состоит из трёх стадий: **свёртка** (линейные активации $v$) → **функция активации** $\varphi$
(выходы $s = \varphi(v)$) → **пространственное объединение** (pooling) $p$.

**Свёртка.** Как и в большинстве библиотек, `nn.Conv2d` вычисляет кросс-корреляцию. Для одноканального входа $I$ и ядра $K$:

$$
s(i, j) = \langle I * K \rangle(i, j) = \sum_{m}\sum_{n} I(i + m, j + n)\, K(m, n).
$$

Для многоканального входа ($k_{i-1}$ каналов) каждое из $k_i$ ядер слоя — трёхмерная матрица $k_{i-1} \times wc_i \times hc_i$,
и сумма берётся ещё и по каналам $c$; выход — $k_i$ карт признаков:

$$
v_{c'}(i, j) = \sum_{c=1}^{k_{i-1}} \sum_{m=0}^{wc_i - 1} \sum_{n=0}^{hc_i - 1} I_c(i + m, j + n)\, K_{c'}(c, m, n) \;\big(+\, b_{c'}\big),
\qquad c' = \overline{1, k_i}.
$$

**Размер карты признаков.** В лекции: $w_i = \left[\dfrac{w_{i-1} - wc_i}{sc_i}\right] + 1$. У нас вход дополняется рамкой ширины
$pad_i$, поэтому

$$
w_i = \left[\frac{w_{i-1} + 2\,pad_i - wc_i}{sc_i}\right] + 1, \qquad
h_i = \left[\frac{h_{i-1} + 2\,pad_i - hc_i}{sc_i}\right] + 1 .
$$

Для всех свёрток сети $wc_i = hc_i = 3$, $sc_i = 1$, $pad_i = 1$ $\Rightarrow$ $w_i = w_{i-1}$: свёртка размер не меняет.
Пространственное объединение по максимуму с окном $wp_i \times hp_i = 2\times2$ и шагом $sp_i = 2$:
$\left[\frac{w_i - 2}{2}\right] + 1 = w_i / 2$ — уменьшает карту вдвое.

**Объединение по максимуму** $p = \max_{i = \overline{1, q}} x_i$, $\ \dfrac{\partial p}{\partial x_i} = 1$ для максимального элемента и 0 для остальных:
при обратном проходе ошибка $\delta^{(pool)}$ передаётся только тому нейрону окна, на выходе которого было максимальное значение.

**Обратный проход** через сверточный слой (лекция): $\delta^{(conv)}_j = \delta^{(pool)}\,\dfrac{\partial p}{\partial s_j}\dfrac{\partial \varphi}{\partial v_j}$,
$\ \dfrac{\partial E}{\partial w} = \langle x * \delta^{(conv)} \rangle$ — градиент по ядру есть свёртка входа с локальными градиентами.
Эти производные для всех слоёв PyTorch вычисляет автоматически при вызове `loss.backward()`.

### 4.2. Сеть ResNet-9

Базовый блок — `ConvBlock`: свёртка $3\times3$ → BatchNorm → ReLU ($\varphi$), при необходимости — MaxPool $2\times2$.

| Этап | Слои | Выход $k_i \times w_i \times h_i$ |
|---|---|---|
| вход | — | $k_0 \times w \times h = 3 \times 32 \times 32$ |
| prep | ConvBlock(3 → 64) | 64 × 32 × 32 |
| layer1 | ConvBlock(64 → 128) + MaxPool | 128 × 16 × 16 |
| res1 | Residual: 2 × ConvBlock(128 → 128) | 128 × 16 × 16 |
| layer2 | ConvBlock(128 → 256) + MaxPool | 256 × 8 × 8 |
| layer3 | ConvBlock(256 → 512) + MaxPool | 512 × 4 × 4 |
| res2 | Residual: 2 × ConvBlock(512 → 512) | 512 × 4 × 4 |
| head | глобальный MaxPool → Flatten → Linear(512 → 10) → × 0.125 | $M = 10$ |

Почему так:
- **Свёртки $3\times3$** (как в VGG; принцип из лекции — заменять свёртку большого размера стеком свёрток меньшего):
  два слоя $3\times3$ имеют то же рецептивное поле, что один $5\times5$, но $2 \cdot 9 = 18$ весов на пару каналов вместо 25 и больше нелинейностей.
- **BatchNorm** нормирует линейные активации каждого канала по пачке:
  $\hat v = \dfrac{v - \mu_{\mathcal{B}}}{\sqrt{\sigma^2_{\mathcal{B}} + \epsilon}}$, $\ v' = \gamma \hat v + \beta$ ($\gamma, \beta$ обучаются).
  Обучение устойчиво при большой скорости обучения, есть слабый регуляризующий эффект. Поэтому у свёрток перед BN нет смещения
  (`bias=False`) — оно всё равно поглощается параметром $\beta$.
- **Остаточные связи** $x^{out} = x + F(x)$: $\dfrac{\partial E}{\partial x} = \dfrac{\partial E}{\partial x^{out}}\left(I + \dfrac{\partial F}{\partial x}\right)$ —
  градиент проходит через тождественную ветвь напрямую, даже если $\partial F / \partial x$ мал; блоку достаточно выучить «поправку» $F(x)$.
- **Число каналов $k_i$ растёт** при уменьшении $w_i \times h_i$: чем глубже, тем более абстрактные и многочисленные признаки,
  а пространственное объединение выполняется по картам меньшей размерности (ещё один принцип из лекции).
- **Глобальный пулинг** вместо развёртки карты $512\times4\times4$ в вектор: в 16 раз меньше параметров в классификаторе.
- **Масштаб выходов 0.125** (приём из оригинального ResNet-9, DAWNBench). После глобального max-пулинга признаки — максимумы
  по $4\times4$ положительных ReLU-активаций, поэтому при случайной инициализации линейные активации выходного слоя $v_m$ (логиты)
  получаются большими (порядка единиц–десятков), softmax сразу насыщается, и первые шаги SGD с $\eta=0.1$ «взрывают» функцию ошибки.
  Умножение на $1/8$ делает начальные $v_m$ порядка 1 (начальная ошибка ≈ $\ln 10 \approx 2.3$). Это эквивалентно уменьшению
  начальных весов и скорости обучения последнего слоя. Проверено экспериментально: без масштаба функция ошибки за первые шаги
  вырастала до ~100, и сеть не обучалась.
- Выход сети — **линейные активации $v_m$** (без softmax): `nn.CrossEntropyLoss` сама вычисляет $u_m = \operatorname{softmax}(v)_m$ и
  $\ln u_m$ через log-sum-exp — это численно устойчивее.

In [ ]:
class ConvBlock(nn.Module):
    # Conv3x3 -> BatchNorm -> ReLU [-> MaxPool2x2]
    def __init__(self, in_ch, out_ch, pool=False):
        super().__init__()
        self.conv = nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1, bias=False)
        self.bn = nn.BatchNorm2d(out_ch)
        self.pool = nn.MaxPool2d(2) if pool else nn.Identity()

    def forward(self, x):
        return self.pool(F.relu(self.bn(self.conv(x)), inplace=True))


class ResidualBlock(nn.Module):
    # x_out = x + F(x), F — две последовательные ConvBlock с тем же числом каналов
    def __init__(self, ch):
        super().__init__()
        self.block1 = ConvBlock(ch, ch)
        self.block2 = ConvBlock(ch, ch)

    def forward(self, x):
        return x + self.block2(self.block1(x))


class ResNet9(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.prep = ConvBlock(3, 64)
        self.layer1 = ConvBlock(64, 128, pool=True)
        self.res1 = ResidualBlock(128)
        self.layer2 = ConvBlock(128, 256, pool=True)
        self.layer3 = ConvBlock(256, 512, pool=True)
        self.res2 = ResidualBlock(512)
        self.pool = nn.AdaptiveMaxPool2d(1)
        self.fc = nn.Linear(512, num_classes)
        self.logit_scale = 0.125   # см. пояснение выше: без масштаба обучение на старте расходится

    def forward(self, x):
        x = self.prep(x)
        x = self.res1(self.layer1(x))
        x = self.layer2(x)
        x = self.res2(self.layer3(x))
        x = torch.flatten(self.pool(x), 1)
        return self.fc(x) * self.logit_scale   # линейные активации выходного слоя v_m


model = ResNet9().to(DEVICE).to(memory_format=torch.channels_last)
print(model)
print(f"\nЧисло обучаемых параметров: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

In [ ]:
# проверка размерностей на каждом этапе
with torch.no_grad():
    x = torch.zeros(2, 3, 32, 32, device=DEVICE)
    for name, layer in model.named_children():
        if name == "fc":
            x = torch.flatten(x, 1)
        x = layer(x)
        print(f"{name:7s} -> {tuple(x.shape)}")

### 4.3. Количество обучаемых параметров

По формуле из лекции свёрточные слои содержат $\sum_{i=1}^{N} (k_{i-1} \cdot wc_i \cdot hc_i + 1) \cdot k_i$ параметров,
где «+1» — смещение каждого ядра. В нашей сети у свёрток смещения нет (`bias=False`), зато после каждой свёртки стоит BatchNorm
с двумя параметрами $\gamma, \beta$ на канал, поэтому слой $i$ даёт $k_{i-1} \cdot wc_i \cdot hc_i \cdot k_i + 2 k_i$.
Полносвязный слой: $(512 + 1) \cdot M$. Слои пулинга и масштаб 0.125 параметров не имеют. Сверим формулу с подсчётом PyTorch:

In [ ]:
# (k_{i-1}, k_i) для N = 8 сверточных слоёв, wc_i = hc_i = 3
conv_channels = [(3, 64), (64, 128), (128, 128), (128, 128), (128, 256), (256, 512), (512, 512), (512, 512)]
wc = hc = 3
n_conv = sum(k_prev * wc * hc * k for k_prev, k in conv_channels)
n_bn = sum(2 * k for _, k in conv_channels)
n_fc = (512 + 1) * 10
print(f"свёртки:   {n_conv:>10,}")
print(f"BatchNorm: {n_bn:>10,}")
print(f"FC:        {n_fc:>10,}")
print(f"итого по формуле: {n_conv + n_bn + n_fc:,}")
print(f"итого PyTorch:    {sum(p.numel() for p in model.parameters()):,}")

## 5. Функция ошибки, оптимизатор и расписание скорости обучения

**Функция ошибки — кросс-энтропия.** Выход сети $u_m = \dfrac{e^{v_m}}{\sum_{l=1}^{M} e^{v_l}}$, $m = \overline{1, M}$, $M = 10$ —
распределение вероятностей по классам. Каждое изображение относится ровно к одному классу, поэтому (как и в лабораторной работе №1)
используется категориальная кросс-энтропия:

$$
\min_w E(w) = \min_w \left\{ -\frac{1}{L} \sum_{k=1}^{L} \sum_{m=1}^{M} \tilde y^k_m \ln u^k_m \right\}, \qquad L = 50\,000 .
$$

**Сглаживание меток (label smoothing, $\varepsilon = 0.1$).** Вместо one-hot меток $y^k_m$ используются

$$
\tilde y^k_m = (1 - \varepsilon)\, y^k_m + \frac{\varepsilon}{M}
\quad\Rightarrow\quad
\tilde y^k_m = 0.91 \text{ для верного класса и } 0.01 \text{ для остальных.}
$$

С one-hot метками минимум $E$ достигается только при $u_m \to 1$, т.е. $v_m \to \infty$: сеть стремится к бесконечно уверенным
ответам и переобучается. Со сглаженными метками оптимум конечен; на практике это даёт +0.3–0.5% точности.

**Оптимизатор — SGD с моментом Нестерова** ($\mu = 0.9$) **и weight decay** ($\lambda = 5\cdot10^{-4}$, L2-регуляризация весов).
На шаге $t$ по пачке $\mathcal{B}$ вычисляется $g(t) = \dfrac{\partial E_{\mathcal{B}}}{\partial w} + \lambda w(t)$ и

$$
\Delta w(t) = \mu\, \Delta w(t-1) + g(t), \qquad
w(t+1) = w(t) - \eta(t)\, \big(g(t) + \mu\, \Delta w(t)\big).
$$

Момент накапливает направление спуска и сглаживает шум стохастического градиента; для свёрточных сетей на CIFAR SGD с моментом
обычно обобщает лучше Adam.

**Расписание One-Cycle для $\eta(t)$:** первые 25% шагов скорость обучения растёт от $\eta_{max}/25$ до $\eta_{max} = 0.1$
(разогрев: на старте веса случайны, большой шаг опасен), затем по косинусу падает почти до нуля. Большой шаг в середине — быстрое
продвижение и «шум», помогающий выйти к широким минимумам, маленький в конце — точная донастройка.

**Смешанная точность (AMP):** прямой и обратный проход в float16 на GPU примерно в 2 раза быстрее; `GradScaler` умножает функцию ошибки
на большой множитель перед `backward()`, чтобы маленькие градиенты не обнулялись в float16, и делит градиенты обратно перед шагом.

In [ ]:
criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)
optimizer = torch.optim.SGD(model.parameters(), lr=MAX_LR, momentum=MOMENTUM,
                            nesterov=True, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.OneCycleLR(optimizer, max_lr=MAX_LR, epochs=EPOCHS,
                                                steps_per_epoch=len(train_loader), pct_start=0.25)
scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)

lrs = []
_opt = torch.optim.SGD([torch.zeros(1, requires_grad=True)], lr=MAX_LR)
_sch = torch.optim.lr_scheduler.OneCycleLR(_opt, max_lr=MAX_LR, epochs=EPOCHS, steps_per_epoch=len(train_loader), pct_start=0.25)
for _ in range(EPOCHS * len(train_loader)):
    lrs.append(_sch.get_last_lr()[0]); _opt.step(); _sch.step()
plt.figure(figsize=(6, 2.5))
plt.plot(np.arange(len(lrs)) / len(train_loader), lrs)
plt.xlabel("эпоха"); plt.ylabel("learning rate"); plt.title("Расписание One-Cycle"); plt.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 6. Обучение

После каждой эпохи сеть переводится в режим `eval()` (BatchNorm использует накопленную статистику) и оценивается
на **всей обучающей выборке без аугментации**: выводится функция ошибки (кросс-энтропия), ошибка классификации (доля неверных ответов) и точность.
Для построения графиков на каждой эпохе также считается качество на тестовой выборке; ни ранняя остановка, ни выбор модели по нему не делаются —
итоговая модель — это модель после последней эпохи.

In [ ]:
@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    total_loss, correct, n = 0.0, 0, 0
    for x, y in loader:
        x = x.to(DEVICE, non_blocking=True).to(memory_format=torch.channels_last)
        y = y.to(DEVICE, non_blocking=True)
        with torch.autocast(DEVICE.type, enabled=USE_AMP):
            logits = model(x)
        total_loss += F.cross_entropy(logits.float(), y, reduction="sum").item()   # без сглаживания — «честная» ошибка
        correct += (logits.argmax(1) == y).sum().item()
        n += len(y)
    return total_loss / n, correct / n


def train_one_epoch(model, loader):
    model.train()
    for x, y in loader:
        x = x.to(DEVICE, non_blocking=True).to(memory_format=torch.channels_last)
        y = y.to(DEVICE, non_blocking=True)
        with torch.autocast(DEVICE.type, enabled=USE_AMP):
            loss = criterion(model(x), y)
        optimizer.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()      # обратное распространение: dE/dw для всех слоёв (autograd)
        scaler.step(optimizer)             # шаг SGD
        scaler.update()
        scheduler.step()                   # One-Cycle меняет lr после каждого шага

In [ ]:
history = {k: [] for k in ("train_loss", "train_acc", "test_loss", "test_acc", "time")}
t_total = time.perf_counter()
for epoch in range(1, EPOCHS + 1):
    t0 = time.perf_counter()
    train_one_epoch(model, train_loader)
    train_time = time.perf_counter() - t0
    tr_loss, tr_acc = evaluate(model, train_eval_loader)
    te_loss, te_acc = evaluate(model, test_loader)
    for k, v in zip(history, (tr_loss, tr_acc, te_loss, te_acc, train_time)):
        history[k].append(v)
    print(f"Эпоха {epoch:2d}/{EPOCHS} | train: loss {tr_loss:.4f}, ошибка {1 - tr_acc:.4f}, точность {tr_acc:.2%} "
          f"| test: точность {te_acc:.2%} | lr {scheduler.get_last_lr()[0]:.4f} | {train_time:.1f} с")
print(f"\nОбщее время обучения: {(time.perf_counter() - t_total) / 60:.1f} мин")

## 7. Оценка качества на тестовой выборке

In [ ]:
test_loss, test_acc = evaluate(model, test_loader)
print(f"Тестовая выборка: функция ошибки = {test_loss:.4f}")
print(f"                  ошибка классификации = {1 - test_acc:.4f}")
print(f"                  точность = {test_acc:.2%}")

In [ ]:
ep = np.arange(1, EPOCHS + 1)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(ep, history["train_loss"], "o-", label="train")
ax1.plot(ep, history["test_loss"], "o-", label="test")
ax1.set_xlabel("эпоха"); ax1.set_ylabel("кросс-энтропия"); ax1.set_title("Функция ошибки")
ax1.legend(); ax1.grid(alpha=0.3)
ax2.plot(ep, 100 * (1 - np.array(history["train_acc"])), "o-", label="train")
ax2.plot(ep, 100 * (1 - np.array(history["test_acc"])), "o-", label="test")
ax2.set_xlabel("эпоха"); ax2.set_ylabel("ошибка классификации, %"); ax2.set_title("Ошибка классификации")
ax2.set_yscale("log"); ax2.legend(); ax2.grid(alpha=0.3, which="both")
plt.tight_layout()
plt.show()

In [ ]:
@torch.no_grad()
def predict(model, loader):
    model.eval()
    preds = []
    for x, _ in loader:
        x = x.to(DEVICE).to(memory_format=torch.channels_last)
        with torch.autocast(DEVICE.type, enabled=USE_AMP):
            preds.append(model(x).argmax(1).cpu())
    return torch.cat(preds).numpy()


y_true = np.array(test_ds.targets)
y_pred = predict(model, test_loader)

cm = np.zeros((10, 10), dtype=int)
np.add.at(cm, (y_true, y_pred), 1)
print("Точность по классам:")
for c, name in enumerate(CLASSES):
    print(f"  {name:11s} {cm[c, c] / cm[c].sum():.2%}")

fig, ax = plt.subplots(figsize=(7, 6))
ax.imshow(cm, cmap="Blues")
for i in range(10):
    for j in range(10):
        ax.text(j, i, cm[i, j], ha="center", va="center", fontsize=8,
                color="white" if cm[i, j] > cm.max() / 2 else "black")
ax.set_xticks(range(10), CLASSES, rotation=45, ha="right"); ax.set_yticks(range(10), CLASSES)
ax.set_xlabel("предсказание"); ax.set_ylabel("истинный класс"); ax.set_title("Матрица ошибок (test)")
plt.tight_layout()
plt.show()

In [ ]:
wrong = np.flatnonzero(y_pred != y_true)[:16]
fig, axes = plt.subplots(2, 8, figsize=(13, 3.8))
for ax, i in zip(axes.flat, wrong):
    ax.imshow(test_raw.data[i])
    ax.set_title(f"{CLASSES[y_true[i]]}\n→ {CLASSES[y_pred[i]]}", fontsize=9, color="C3")
    ax.axis("off")
plt.suptitle("Ошибки на тестовой выборке (истина → предсказание)")
plt.tight_layout()
plt.show()

### Усреднение по отражению (Test-Time Augmentation)

Сеть обучалась на изображениях и их зеркальных копиях, поэтому для каждой тестовой картинки можно получить два
предсказания — для исходной и для отражённой — и усреднить вероятности. Ошибки на двух вариантах частично независимы,
и усреднение обычно даёт +0.2–0.5% точности. Обучение при этом не меняется; основной результат работы — точность без TTA
(выше), здесь — дополнительный.

In [ ]:
@torch.no_grad()
def evaluate_tta(model, loader):
    model.eval()
    correct, n = 0, 0
    for x, y in loader:
        x = x.to(DEVICE).to(memory_format=torch.channels_last)
        y = y.to(DEVICE)
        with torch.autocast(DEVICE.type, enabled=USE_AMP):
            probs = model(x).float().softmax(1) + model(torch.flip(x, dims=[3])).float().softmax(1)
        correct += (probs.argmax(1) == y).sum().item()
        n += len(y)
    return correct / n


tta_acc = evaluate_tta(model, test_loader)
print(f"Точность на тесте без TTA:         {test_acc:.2%}")
print(f"Точность на тесте с TTA (+ flip):  {tta_acc:.2%}")

In [ ]:
torch.save(model.state_dict(), "resnet9_cifar10.pt")
print("Веса сохранены в resnet9_cifar10.pt")

## 8. Выводы

1. Реализована собственная свёрточная сеть в стиле ResNet-9 (классы `ConvBlock`, `ResidualBlock`, `ResNet9`) из базовых слоёв PyTorch:
   `Conv2d`, `BatchNorm2d`, `ReLU`, `MaxPool2d`, `AdaptiveMaxPool2d`, `Linear`.
2. Данные нормализованы по статистике обучающей выборки; для борьбы с переобучением использованы аугментация
   (случайный сдвиг, отражение, Cutout), label smoothing и weight decay.
3. Обучение — SGD с моментом Нестерова и расписанием One-Cycle; BatchNorm и остаточные связи позволяют использовать большую
   скорость обучения и сходиться за небольшое число эпох.
4. Итоговые ошибка и точность на тестовой выборке выведены в разделе 7, там же — прирост от усреднения по отражению (TTA); по матрице ошибок видно, что больше всего путаются
   визуально близкие классы (кошка/собака, автомобиль/грузовик, птица/самолёт).